## Structure of the notebook
| Cell | Content                                                                  |
|------|--------------------------------------------------------------------------|
| 1    | Imports                                                                  |
| 2    | General configuration                                                    |
| 3    | Text normalisation and tokenisation                                      |
| 4    | Function word filtering and stopword construction                        |
| 5    | Lexicon loading and diagnostics                                          |
| 6    | Corpus loading and chunking functions                                    |
| 7    | Statistical utility functions                                            |
| 8    | Shared resources (spaCy - sentence embeddings - sentiment)               |
| 9    | BERTopic modeling + Q1 scoring (quantitative analysis)                   |
| 9b   | Q1 comparative summary tables (Excel export)                             |
| 10   | Q2 — Qualitative differential analysis (1: Topics - 2: TF-IDF -         |
|      | 3: Co-occurrences - 4: Emotional valence)                                |
| 11   | Q2 — Per-corpus summary tables (Excel export)                            |

In [1]:
# 1 - Librairies

import os
import re
import glob
import json
import random
import warnings
import unicodedata
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Set, Tuple
from collections import Counter

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.stats import mannwhitneyu, chi2_contingency
from scipy.spatial.distance import jensenshannon
from statsmodels.stats.multitest import multipletests

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

from functionwordsets import load as fw_load
from transformers import pipeline as hf_pipeline, CamembertTokenizer
import spacy

print('Imports OK')

/home/vsaiag/Desktop/article1_test/.venv-1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
# 2 - General configuration

# REPRODUCTIBILITY
# Fixing all random seeds to ensure reproducibility across runs

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# CORPUS
# 4 authorial categories of corpora, each subdivided into prose and poetry
CORPUS_CONFIGS = [
    {
        'name': 'lesbiennes',
        'path_poetry': './corpi_these/corpus_poesie_lesb',
        'path_prose':  './corpi_these/corpus_lesbiennes',
    },
    {
        'name': 'f_hetero',
        'path_poetry': './corpi_these/corpus_poesie_hetero',
        'path_prose':  './corpi_these/corpus_hetero',
    },
    {
        'name': 'homo_m',
        'path_poetry': './corpi_these/corpus_poesie_homosexuels',
        'path_prose':  './corpi_these/corpus_homosexuels',
    },
    {
        'name': 'h_hetero',
        'path_poetry': './corpi_these/corpus_poesie_homhet',
        'path_prose':  './corpi_these/corpus_homhet',
    },
]

FILE_PATTERN = '*.txt'

# OUTLIER REDUCTION (post fit_transform)
# We used 3 strategies to reassign chunks that HDBSCAN initially assignent to topic -1 (see cell 9)
REDUCE_OUTLIERS            = True
REDUCE_OUTLIERS_STRATEGIES = ('embeddings', 'c-tf-idf')
REASSIGN_BY_CENTROID       = True
REASSIGN_MIN_SIMILARITY    = 0.30 # threshold of minimal cosine similarity
TEMPORAL_FILL              = True # reassigning an outlier to a topic if it is positionned between 2 chunks of the same topic

# CHUNKING
# Chunking into segments of 300 words with no overlap 
CHUNK_SIZE           = 300
CHUNK_STRIDE         = 300
MIN_TOKENS_PER_CHUNK = 80

# EMBEDDINGS
EMBED_MODEL_NAME = 'sentence-transformers/distiluse-base-multilingual-cased-v1'
BATCH_SIZE_GPU   = 128
BATCH_SIZE_CPU   = 32

# SPACY
SPACY_MODEL      = 'fr_core_news_md'
SPACY_BATCH_SIZE = 256

# LEMMATISATION MATCHING
CONTEXT_LEMMA_MIN_SHARE = float(os.getenv('CONTEXT_LEMMA_MIN_SHARE', '0.55'))
# = minimum consistency rate for contextual lemma assignment 
# (a surface form is mapped to its most frequent lemma only if
# that lemma accounts for at least 55% of observed occurrences)

CONTEXT_LEMMA_MIN_COUNT = int(os.getenv('CONTEXT_LEMMA_MIN_COUNT', '2'))
# = minimum number of corpus occurrences required before contextual lemmatisation is applied

LEXICON_MATCHING_MODES  = ['lemma_only', 'lemma_plus_surface_backoff']
LEXICON_PATH            = os.getenv('LEXICON_PATH', './lexicon_master.xlsx')

# OUTPUTS
OUT_DIR_ROOT = os.getenv('OUT_DIR', './final_results')

# BERTOPIC
BER_TOPIC_TOP_N_WORDS = 120
RUNS_TOPN             = {'A': 20, 'B': 40, 'C': 60}

# Q2 - DIFFERENTIAL ANALYSIS 
TARGET_LEXICONS = None 

# A chunk is retained for Q2 analysis only if its direct lexical score exceeds this threshold 
# (= at least one lexicon lemma is present in the chunk)
SCORE_THRESHOLD_DIRECT = 0.0

TFIDF_TOP_N  = 30
COOC_WINDOW  = 10

# Reference run for Q2
RUN_ID_REF = 'B'
MODE_REF   = 'lemma_only'

# SENTIMENT ANALYSIS
SENTIMENT_BATCH_SIZE = 32
SENTIMENT_MAX_CHARS  = 1800
SENTIMENT_MODEL_NAME = 'cmarkea/distilcamembert-base-sentiment'

print('Configuration OK')

Configuration OK


In [3]:
# 3 - Text normalisation and tokenisation

TOKEN_RE = re.compile(r"\b[\w'-]+\b", flags=re.UNICODE)

def strip_accents(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, normalise typographic apostrophes, collapse whitespace
def normalize_text(text: str) -> str:
    text = text.lower()
    text = text.replace('\u2019', "'")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def norm_token(t: str) -> str:
    return strip_accents(normalize_text(t))

def tokenize(text: str) -> List[str]:
    return TOKEN_RE.findall(normalize_text(text))

print('Normalisation OK')

Normalisation OK


In [4]:
# 4 - Function word filtering and stopword construction 

fw = fw_load('fr_21c')

SELECTED_FW_CATEGORIES = [
    'articles', 'poss_det', 'pers_pron', 'poss_pron', 'dem_pron',
    'indef_pron', 'inter_pron', 'prepositions', 'coord_conj',
    'subord_conj', 'negations', 'aux_être', 'aux_avoir', 'modals_full',
]

missing_fw = [c for c in SELECTED_FW_CATEGORIES if c not in fw.categories]
if missing_fw:
    raise KeyError(f'Unknown functionwordsets categories: {missing_fw}')

fw_selected = fw.subset(SELECTED_FW_CATEGORIES)

FW_SINGLE: Set[str] = set()
FW_MULTI:  Set[str] = set()
for w in fw_selected:
    w_norm = norm_token(w)
    if not w_norm:
        continue
    if ' ' in w_norm:
        FW_MULTI.add(w_norm)
    else:
        FW_SINGLE.add(w_norm)

PROJECT_NOISE    = {norm_token(x) for x in {'wikisource', 'gallica', 'http', 'https', 'www', 'licence'}}
STOPWORDS_SINGLE = FW_SINGLE | PROJECT_NOISE
EXTRA_STOPWORDS = {
    'est', 'etait', 'sera', 'serait', 'soit', 'ete', 'etre',
    'ont', 'avait', 'aura', 'aurait', 'ait',
    'bien', 'meme', 'tout', 'plus', 'tres', 'aussi',
    'car', 'donc', 'puis', 'ainsi', 'comme',
    'quand', 'si', 'non', 'qu', 'dit', 'voici', 'peu'
}
STOPWORDS_SINGLE = STOPWORDS_SINGLE | {norm_token(w) for w in EXTRA_STOPWORDS}
# STOPWORDS_SINGLE set is passed to the BERTopic CountVectorizer to prevent function words from dominating topic representations
STOPWORDS_MULTI  = FW_MULTI

def remove_multiword_function_phrases(text: str, multi_phrases: Set[str]) -> str:
    txt = norm_token(text)
    for phrase in sorted(multi_phrases, key=len, reverse=True):
        pattern = r'(?<!\w)' + re.escape(phrase) + r'(?!\w)'
        txt = re.sub(pattern, ' ', txt)
    return re.sub(r'\s+', ' ', txt).strip()

def clean_for_vectorizer(text: str) -> str:
    txt = remove_multiword_function_phrases(text, STOPWORDS_MULTI)
    toks = tokenize(txt)
    toks = [norm_token(t) for t in toks]
    toks = [t for t in toks if t and t not in STOPWORDS_SINGLE and len(t) > 1 and not t.isdigit()]
    return ' '.join(toks)

print('Function words OK')

Function words OK


In [5]:
# 5 - Lexicon loading and diagnostics

REQUIRED_LEXICON_COLS = {'lexicon', 'lemma', 'multiword'}

def load_lexicons(path: str):
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Lexicon file not found: {path}')
    lx = pd.read_excel(path) if path.lower().endswith('.xlsx') else pd.read_csv(path)
    missing_cols = REQUIRED_LEXICON_COLS - set(lx.columns)
    if missing_cols:
        raise ValueError(f'Missing lexicon columns: {missing_cols}')
    lx['lexicon'] = lx['lexicon'].astype(str).str.strip()
    lx['lemma']   = lx['lemma'].astype(str).map(norm_token)
    mw_num = pd.to_numeric(lx['multiword'], errors='coerce')
    if mw_num.notna().any():
        lx['multiword'] = mw_num.fillna(0).astype(int)
    else:
        mw_str = lx['multiword'].astype(str).str.strip().str.lower()
        lx['multiword'] = mw_str.map(
            {'1': 1, 'true': 1, 'yes': 1, 'y': 1,
             '0': 0, 'false': 0, 'no': 0, 'n': 0}
        ).fillna(0).astype(int)
    lx = lx[(lx['lexicon'] != '') & (lx['lemma'] != '')]
    lx_single = lx[lx['multiword'] == 0].copy()
    if lx_single.empty:
        raise ValueError('No single-word lexicon entries found.')
    lex_map = {lex: set(sub['lemma'].tolist()) for lex, sub in lx_single.groupby('lexicon')}
    return lex_map, lx_single

LEXICON_MAP, lexicon_loaded_df = load_lexicons(LEXICON_PATH)
lexicon_names = sorted(LEXICON_MAP.keys())

# TARGET_LEXICONS is set to all loaded lexicons by default, we can exclude one by modifying this list
TARGET_LEXICONS = list(lexicon_names)
print(f'Lexiques chargés  : {lexicon_names}')
print(f'TARGET_LEXICONS   : {TARGET_LEXICONS}')

# Checking for coherence
missing_targets = [l for l in TARGET_LEXICONS if l not in LEXICON_MAP]
if missing_targets:
    raise ValueError(f'TARGET_LEXICONS contains words absent from the loaded lexicon : {missing_targets}')

# Diagnostics : Jaccard overlap
overlap_rows = []
for _i, _a in enumerate(lexicon_names):
    for _b in lexicon_names[_i + 1:]:
        _A, _B = LEXICON_MAP[_a], LEXICON_MAP[_b]
        _inter = _A & _B
        _union = _A | _B
        overlap_rows.append({
            'lexicon_a': _a, 'lexicon_b': _b,
            'n_a': len(_A), 'n_b': len(_B),
            'n_intersection': len(_inter),
            'jaccard': len(_inter) / len(_union) if _union else 0.0
        })
overlap_df = pd.DataFrame(overlap_rows).sort_values('jaccard', ascending=False)
print(overlap_df.head())

Lexiques chargés  : ['lesbian_love', 'love', 'love_and_desire', 'physical_desire', 'sapphism']
TARGET_LEXICONS   : ['lesbian_love', 'love', 'love_and_desire', 'physical_desire', 'sapphism']
         lexicon_a        lexicon_b  n_a  n_b  n_intersection   jaccard
7  love_and_desire  physical_desire   55   52              52  0.945455
0     lesbian_love             love   14    7               7  0.500000
3     lesbian_love         sapphism   14    7               7  0.500000
4             love  love_and_desire    7   55               7  0.127273
1     lesbian_love  love_and_desire   14   55               7  0.112903


In [6]:
# 6 - Corpus loading and chunking functions

# Each text file is tokenised then split into chunks 
# Two text representations are stored per chunk:
# 1) text_raw: original normalised content, used for embeddings and spaCy lemmatisation
# 2) text_vectorizer: function-word-filtered version, used as input to the BERTopic CountVectorizer
# Work and author identifiers are extracted from the filename

@dataclass
class ChunkRecord:
    chunk_id:        str
    work_id:         str
    author_id:       str
    genre:           str
    text_raw:        str
    text_vectorizer: str

def parse_work_and_author_from_filename(path: str) -> Tuple[str, str]:
    stem = os.path.splitext(os.path.basename(path))[0]
    author_id = stem.split('__')[0] if '__' in stem else stem.split('_')[0]
    return stem, author_id

def chunk_tokens(tokens, chunk_size, stride, min_tokens):
    chunks, i = [], 0
    while i < len(tokens):
        ch = tokens[i:i + chunk_size]
        if len(ch) >= min_tokens:
            chunks.append(ch)
        i += stride
    return chunks

def load_corpus(base_path, genre_label, chunk_size, chunk_stride, min_tokens):
    if not os.path.isdir(base_path):
        raise FileNotFoundError(f'Path not found: {base_path}')
    records = []
    files = sorted(glob.glob(os.path.join(base_path, '**', FILE_PATTERN), recursive=True))
    files = [f for f in files if not f.endswith('_lemmatized.txt')]
    print(f'  [{genre_label}] {len(files)} files found in {base_path}')
    for fp in files:
        with open(fp, 'r', encoding='utf-8', errors='ignore') as fh:
            txt = fh.read()
        toks = tokenize(normalize_text(txt))
        work_id, author_id = parse_work_and_author_from_filename(fp)
        for i, ch in enumerate(chunk_tokens(toks, chunk_size, chunk_stride, min_tokens)):
            raw = ' '.join(ch)
            vec = clean_for_vectorizer(raw)
            if len(raw.split()) < min_tokens or not vec:
                continue
            records.append(ChunkRecord(
                f'{work_id}__{i}', work_id, author_id, genre_label, raw, vec))
    return records

def records_to_df(records):
    if not records:
        return pd.DataFrame(columns=[
            'chunk_id', 'work_id', 'author_id', 'genre', 'text_raw', 'text_vectorizer'])
    return pd.DataFrame([r.__dict__ for r in records])

print('Corpus functions OK')

Corpus functions OK


In [7]:
# 7 - Statistical utility functions 

# cliffs_delta: non-parametric effect size estimator. Returns a value in [-1, 1] quantifying 
# the probability that a random observation from group x exceeds a random observation from group y
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) == 0 or len(y) == 0:
        return np.nan
    greater = sum(np.sum(xi > y) for xi in x)
    lower   = sum(np.sum(xi < y) for xi in x)
    return (greater - lower) / (len(x) * len(y))

# compute_tests: applies Mann-Whitney U tests across all lexicons, followed by Holm-Bonferroni correction 
# for multiple comparisons
def compute_tests(
    work_scores: pd.DataFrame,
    lex_names: List[str],
    prefix: str = 'score_'
) -> pd.DataFrame:
    rows = []
    for lex_name in lex_names:
        col = f'{prefix}{lex_name}'
        if col not in work_scores.columns:
            continue
        x = work_scores.loc[work_scores['genre'] == 'poetry', col].to_numpy()
        y = work_scores.loc[work_scores['genre'] == 'prose',  col].to_numpy()
        if len(x) == 0 or len(y) == 0:
            rows.append({'lexicon': lex_name, 'n_poetry': len(x), 'n_prose': len(y),
                         'mean_poetry': np.nan, 'mean_prose': np.nan,
                         'MWU_U': np.nan, 'p_raw': np.nan, 'cliffs_delta': np.nan})
            continue
        U, p = mannwhitneyu(x, y, alternative='two-sided')
        rows.append({'lexicon': lex_name, 'n_poetry': len(x), 'n_prose': len(y),
                     'mean_poetry': float(np.mean(x)), 'mean_prose': float(np.mean(y)),
                     'MWU_U': float(U), 'p_raw': float(p),
                     'cliffs_delta': cliffs_delta(x, y)})
    test_df = pd.DataFrame(rows)
    if test_df.empty:
        return test_df
    test_df['p_holm']           = np.nan
    test_df['significant_0_05'] = False
    mask = test_df['p_raw'].notna().to_numpy()
    if mask.any():
        rej, p_holm, _, _ = multipletests(
            test_df.loc[mask, 'p_raw'], alpha=0.05, method='holm')
        test_df.loc[mask, 'p_holm']           = p_holm
        test_df.loc[mask, 'significant_0_05'] = rej
    return test_df.sort_values(['p_holm', 'p_raw'], na_position='last')

print('Statistical tests OK')

Statistical tests OK


In [8]:
# 8 - Shared resources: spaCy - sentence embeddings - sentiment classifier

# SPACY (contextual lemmatisation)
print('Loading spaCy...')
nlp = spacy.load(SPACY_MODEL, disable=['parser', 'ner', 'textcat'])
print('spaCy OK')

# SENTIMENT EMBEDDINGS
# Embeddings are computed on text_raw (full linguistic context)
# GPU is used when available, the pipeline falls back to CPU automatically
embedding_device   = 'cuda' if torch.cuda.is_available() else 'cpu'
current_batch_size = BATCH_SIZE_GPU if embedding_device == 'cuda' else BATCH_SIZE_CPU
print(f'Embeddings on : {embedding_device}')
embedding_model = SentenceTransformer(EMBED_MODEL_NAME, device=embedding_device)
print('Embedding model OK')

# SENTIMENT CAMEMBERT
print(f'Loading sentiment: {SENTIMENT_MODEL_NAME}...')
_sentiment_tokenizer = CamembertTokenizer.from_pretrained(
    SENTIMENT_MODEL_NAME,
    use_fast=False
)
sentiment_pipeline = hf_pipeline(
    'text-classification',
    model=SENTIMENT_MODEL_NAME,
    tokenizer=_sentiment_tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=512
)
print('Sentiment model OK')

Loading spaCy...
spaCy OK
Embeddings on : cuda


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 264.41it/s]


Embedding model OK
Loading sentiment: cmarkea/distilcamembert-base-sentiment...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 271.97it/s]
CamembertForSequenceClassification LOAD REPORT from: cmarkea/distilcamembert-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model OK


In [9]:
# 9 - BERTopic modelling and Q1 scoring

# Q1 asks whether lesbian authors spoke more about lesbianism/love/desire in poetry or in novel
# To answer this question, for each corpus:
# 1) We load and chunk poetry and prose texts
# 2) We compute sentence embeddings on text_raw (full semantic context)
# 3) We fit BERTopic on text_vectorizer (function-word-filtered)
# 4) We apply the three-stage outlier reduction procedure
# 5) We lemmatise all chunks with spaCy to build chunk_lemma_sets
# 6) We compute score_direct_ per lexicon (independent of topic assignment)
# 7) For each run (A/B/C) × matching mode:
# - we compute topic-affinity scores,
# - we aggregate them at work level,
# - we run Mann-Whitney U tests with Holm-Bonferroni correction,
# - we save our results


for corpus_cfg in CORPUS_CONFIGS:
    corpus_name = corpus_cfg['name']
    OUT_DIR = os.path.join(OUT_DIR_ROOT, corpus_name)
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"Analysis of corpus {corpus_name}")

    # Loading chunks
    poetry_records = load_corpus(
        corpus_cfg['path_poetry'], 'poetry',
        CHUNK_SIZE, CHUNK_STRIDE, MIN_TOKENS_PER_CHUNK)
    prose_records = load_corpus(
        corpus_cfg['path_prose'], 'prose',
        CHUNK_SIZE, CHUNK_STRIDE, MIN_TOKENS_PER_CHUNK)
    all_records = poetry_records + prose_records
    if not all_records:
        print(f'[SKIP] No chunk for {corpus_name}.')
        continue

    df_base = records_to_df(all_records)
    print(f'Total chunks : {len(df_base)}')
    print(df_base['genre'].value_counts())

    # Embeddings (with text_raw)
    try:
        embeddings = embedding_model.encode(
            df_base['text_raw'].tolist(),
            batch_size=current_batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True)
    except RuntimeError as e:
        err = str(e).lower()
        if embedding_device == 'cuda' and ('cuda' in err or 'out of memory' in err):
            print('[Embedding] GPU OOM — bascule CPU...')
            torch.cuda.empty_cache()
            _cpu_model = SentenceTransformer(EMBED_MODEL_NAME, device='cpu')
            embeddings = _cpu_model.encode(
                df_base['text_raw'].tolist(),
                batch_size=BATCH_SIZE_CPU,
                show_progress_bar=True,
                normalize_embeddings=True,
                convert_to_numpy=True)
        else:
            raise

    # BERTopic (with text_vectorizer)
    umap_model = UMAP(
        n_neighbors=5, n_components=5, metric='cosine', random_state=SEED)
    hdbscan_model = HDBSCAN(
        min_cluster_size=10, min_samples=2, metric='euclidean', prediction_data=True)
    vectorizer_model = CountVectorizer(
        stop_words=list(STOPWORDS_SINGLE), ngram_range=(1, 2), min_df=3, max_df=0.95)
    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=True,
        top_n_words=BER_TOPIC_TOP_N_WORDS)

    topics, _ = topic_model.fit_transform(   # [C1]
        df_base['text_vectorizer'].tolist(), embeddings)
    


    # 3-step outlier reduction (post fit_transform)
    # Reassigning chunks that were assignent to topic -1 using 3 strategies:
    topics_arr = np.array(topics, dtype=int)
    outliers_before = int(np.sum(topics_arr == -1))
    print(f'  Outliers before reduction : {outliers_before} '
          f'({100*outliers_before/max(len(topics_arr),1):.1f}%)')

    # 1) BERTopic's built-in reduce_outliers (embedding-based, then c-TF-IDF)
    if REDUCE_OUTLIERS:
        for strategy in REDUCE_OUTLIERS_STRATEGIES:
            if np.sum(topics_arr == -1) == 0:
                break
            try:
                reduced = topic_model.reduce_outliers(
                    documents=df_base['text_vectorizer'].tolist(),
                    topics=topics_arr.tolist(),
                    strategy=strategy,
                    embeddings=embeddings,
                )
                topics_arr = np.array(reduced, dtype=int)
                print(f'  After reduce_outliers ({strategy}): '
                      f'{int(np.sum(topics_arr==-1))} remaining outliers')
            except Exception as e:
                print(f'  [WARN] reduce_outliers ({strategy}) has failed: {e}')

    # 2) Centroid reassignment: each outlier is assigned to the nearest topic centroid by 
    # cosine similarity if the similarity exceeds REASSIGN_MIN_SIMILARITY (default: 0.30)
    if REASSIGN_BY_CENTROID and np.sum(topics_arr == -1) > 0:
        # Calculating centroids bu topic (normalized averages of the embeddings)
        centroids = {}
        for tid in sorted(set(int(t) for t in topics_arr if int(t) != -1)):
            mask = topics_arr == tid
            if not np.any(mask):
                continue
            c = embeddings[mask].mean(axis=0)
            norm = np.linalg.norm(c)
            if norm > 0:
                centroids[tid] = (c / norm).astype(np.float32)

        if centroids:
            outlier_idx = np.where(topics_arr == -1)[0]
            tids  = np.array(sorted(centroids.keys()), dtype=int)
            cmat  = np.vstack([centroids[t] for t in tids])
            emb_out = embeddings[outlier_idx].astype(np.float32)
            norms   = np.linalg.norm(emb_out, axis=1, keepdims=True)
            norms   = np.where(norms == 0, 1e-12, norms)
            emb_out = emb_out / norms
            sims     = emb_out @ cmat.T
            best_idx = np.argmax(sims, axis=1)
            best_sim = sims[np.arange(len(best_idx)), best_idx]
            keep     = best_sim >= REASSIGN_MIN_SIMILARITY
            topics_arr[outlier_idx[keep]] = tids[best_idx[keep]]
            print(f'  After centroid reassignment: '
                  f'{int(np.sum(topics_arr==-1))} remaining outliers')

    # 3) Temporal fill: an outlier chunk between two consecutive chunks of the same topic within 
    # the same work inherits that topic (we assume thematic continuity)
    if TEMPORAL_FILL and np.sum(topics_arr == -1) > 0:
        work_ids = df_base['work_id'].to_numpy()
        n = len(topics_arr)
        for i in range(1, n - 1):
            if topics_arr[i] != -1:
                continue
            if work_ids[i - 1] != work_ids[i] or work_ids[i + 1] != work_ids[i]:
                continue
            left_t  = topics_arr[i - 1]
            right_t = topics_arr[i + 1]
            if left_t != -1 and left_t == right_t:
                topics_arr[i] = left_t
        print(f'  After temporal fill: '
              f'{int(np.sum(topics_arr==-1))} remaining outliers')

    outliers_after = int(np.sum(topics_arr == -1))
    print(f'  Total outliers reduced: {outliers_before - outliers_after} '
          f'({100*(outliers_before-outliers_after)/max(outliers_before,1):.1f}% reassigned)')

    topics = topics_arr.tolist()

    topic_words     = topic_model.get_topics()
    valid_topic_ids = [t for t in topic_words if t != -1]
    print(f'Topics (excl. -1) : {len(valid_topic_ids)}')
    print(f'Proportion outliers (topic=-1) : {float(np.mean(np.array(topics)==-1)):.2%}')

    df_base = df_base.copy()
    df_base['topic'] = topics

    # Contextual spaCy lemmatisation :
    # For each surface form observed in the corpus, the most frequent lemma assigned by spaCy 
    # in context is retained, provided it meets the minimum count (CONTEXT_LEMMA_MIN_COUNT) and 
    # consistency rate (CONTEXT_LEMMA_MIN_SHARE) thresholds. Below these thresholds, we use isolated
    # (single-token) lemmatisation as a fallback. This strategy improves lemma accuracy for archaic or ambiguous forms
    surface_lemma_counts: Dict[str, Counter] = {}
    surface_pos_counts:   Dict[str, Counter] = {}
    chunk_lemma_sets:     List[Set[str]]     = []

    for doc in nlp.pipe(df_base['text_raw'].tolist(), batch_size=SPACY_BATCH_SIZE):
        lemmas_in_chunk: Set[str] = set()
        for t in doc:
            if not t.is_alpha:
                continue
            surf = norm_token(t.text)
            lem  = norm_token(t.lemma_ if t.lemma_ else t.text)
            pos  = (t.pos_ or 'X').upper()
            if surf and lem:
                surface_lemma_counts.setdefault(surf, Counter())[lem] += 1
                surface_pos_counts.setdefault(surf, Counter())[pos]   += 1
                lemmas_in_chunk.add(lem)
        chunk_lemma_sets.append(lemmas_in_chunk)

    surface_best_lemma  = {s: c.most_common(1)[0][0] for s, c in surface_lemma_counts.items() if c}
    surface_best_pos    = {s: c.most_common(1)[0][0] for s, c in surface_pos_counts.items()   if c}
    surface_best_share  = {s: c.most_common(1)[0][1]/sum(c.values()) for s, c in surface_lemma_counts.items() if c}
    surface_total_count = {s: int(sum(c.values())) for s, c in surface_lemma_counts.items()}
    isolated_cache: Dict[str, Tuple[str, str]] = {}

    def analyze_word_better(word: str) -> Tuple[str, str]:
        w = norm_token(word)
        if not w:
            return ('', 'X')
        if (w in surface_best_lemma
                and surface_total_count.get(w, 0) >= CONTEXT_LEMMA_MIN_COUNT
                and surface_best_share.get(w, 0.0) >= CONTEXT_LEMMA_MIN_SHARE):
            return (surface_best_lemma[w], surface_best_pos.get(w, 'X'))
        if w in isolated_cache:
            return isolated_cache[w]
        doc = nlp(w)
        t   = doc[0] if len(doc) else None
        res = (norm_token(t.lemma_ if t and t.lemma_ else word) or w,
               (t.pos_ if t else 'X').upper())
        isolated_cache[w] = res
        return res

    def topic_affinity(topic_id, lex_set, lex_match_set, topn) -> float:
        """Score topic-affinity for a given lexicon"""
        if topic_id not in topic_words or topic_id == -1:
            return 0.0
        s = 0.0
        for w, wt in topic_words[topic_id][:topn]:
            lemma, _ = analyze_word_better(w)
            if lemma in lex_set or norm_token(w) in lex_match_set:
                s += float(wt)
        return s

    def build_surface_backoff_lexicon(lex_set):
        out = set(lex_set)
        for surf, lem in surface_best_lemma.items():
            if lem not in lex_set:
                continue
            if surface_total_count.get(surf, 0) < CONTEXT_LEMMA_MIN_COUNT:
                continue
            if surface_best_share.get(surf, 0.0) < CONTEXT_LEMMA_MIN_SHARE:
                continue
            out.add(surf)
        return out

    def build_lexicon_match_map(mode):
        if mode == 'lemma_plus_surface_backoff':
            return {n: build_surface_backoff_lexicon(e) for n, e in LEXICON_MAP.items()}
        return {n: set(e) for n, e in LEXICON_MAP.items()}

    # Direct lexical score (run- and topic-independent)
    # score_direct_{lex} = |chunk_lemmas ∩ lexicon| / |chunk_lemmas|
    # This score is computed independently of BERTopic topic assignment.
    # It serves as both a robustness check for Q1 and a filtering criterion for Q2
    for lex_name, lex_entries in LEXICON_MAP.items():
        df_base[f'score_direct_{lex_name}'] = [
            len(ls & lex_entries) / max(len(ls), 1)
            for ls in chunk_lemma_sets
        ]

    # Exporting our results 
    topic_model.get_topic_info().to_csv(
        os.path.join(OUT_DIR, 'bertopic_topic_info.csv'), index=False)
    lexicon_loaded_df.to_csv(
        os.path.join(OUT_DIR, 'lexicon_loaded_singleword.csv'), index=False)
    overlap_df.to_csv(
        os.path.join(OUT_DIR, 'lexicon_overlap_diagnostics.csv'), index=False)

    topic_words_rows = []
    for t_id, wws in topic_words.items():
        if t_id == -1:
            continue
        for rank, (w, wt) in enumerate(wws, start=1):
            lem, pos = analyze_word_better(w)
            topic_words_rows.append({'topic': t_id, 'rank': rank, 'word': w,
                                     'word_lemma': lem, 'word_pos': pos, 'weight': wt})
    pd.DataFrame(topic_words_rows).to_csv(
        os.path.join(OUT_DIR, 'bertopic_topic_words_with_lemma.csv'), index=False)

    # Runs A/B/C
    ablation_summary_rows = []
    for matching_mode in LEXICON_MATCHING_MODES:
        LEXICON_MATCH_MAP = build_lexicon_match_map(matching_mode)
        for run_id, topn in RUNS_TOPN.items():
            run_dir = os.path.join(
                OUT_DIR, f'mode_{matching_mode}', f'run_{run_id}_topn_{topn}')
            os.makedirs(run_dir, exist_ok=True)
            df_run = df_base.copy()  

            # Score topic-affinity
            topic_affinity_map = {
                lex_name: {
                    t: topic_affinity(t, lex_e, LEXICON_MATCH_MAP[lex_name], topn)
                    for t in valid_topic_ids
                }
                for lex_name, lex_e in LEXICON_MAP.items()
            }
            for lex_name in lexicon_names:
                df_run[f'score_{lex_name}'] = (
                    df_run['topic'].map(topic_affinity_map[lex_name]).fillna(0.0))

            score_cols  = [c for c in df_run.columns if c.startswith('score_')]
            work_scores = df_run.groupby(
                ['work_id', 'author_id', 'genre'], as_index=False)[score_cols].mean()

            # Tests Q1 - topic-affinity
            test_df_affinity = compute_tests(work_scores, lexicon_names, prefix='score_')
            # Tests Q1 - direct score 
            test_df_direct   = compute_tests(work_scores, lexicon_names, prefix='score_direct_')

            df_run.to_csv(os.path.join(run_dir, 'chunk_level_scores.csv'), index=False)
            work_scores.to_csv(os.path.join(run_dir, 'work_level_scores.csv'), index=False)
            test_df_affinity.to_csv(
                os.path.join(run_dir, 'genre_tests_topic_affinity.csv'), index=False)
            test_df_direct.to_csv(
                os.path.join(run_dir, 'genre_tests_direct.csv'), index=False)

            tmp = test_df_affinity.copy()
            tmp['matching_mode'] = matching_mode
            tmp['run_id']        = run_id
            tmp['topn']          = topn
            ablation_summary_rows.append(tmp)

    ablation_summary_df = pd.concat(ablation_summary_rows, ignore_index=True)
    ablation_summary_df.to_csv(
        os.path.join(OUT_DIR, 'ablation_summary_long.csv'), index=False)

    print(f'[{corpus_name}] Main loop results saved in {os.path.abspath(OUT_DIR)}')

print('\n All corpora have been analyzed for Q1')


Analysis of corpus lesbiennes
  [poetry] 41 files found in ./corpi_these/corpus_poesie_lesb
  [prose] 109 files found in ./corpi_these/corpus_lesbiennes
Total chunks : 15828
genre
prose     14607
poetry     1221
Name: count, dtype: int64


Batches: 100%|██████████| 124/124 [00:16<00:00,  7.35it/s]
2026-04-27 13:02:34,123 - BERTopic - WARNING: Note that extracting more than 100 words from a sparse can slow down computation quite a bit.
2026-04-27 13:02:34,132 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-27 13:02:58,739 - BERTopic - Dimensionality - Completed ✓
2026-04-27 13:02:58,741 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-27 13:02:59,744 - BERTopic - Cluster - Completed ✓
2026-04-27 13:02:59,752 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-27 13:03:07,169 - BERTopic - Representation - Completed ✓


  Outliers before reduction : 6175 (39.0%)
  After reduce_outliers (embeddings): 0 remaining outliers
  Total outliers reduced: 6175 (100.0% reassigned)
Topics (excl. -1) : 368
Proportion outliers (topic=-1) : 0.00%
[lesbiennes] Main loop results saved in /home/vsaiag/Desktop/article1_test/final_results/lesbiennes
Analysis of corpus f_hetero
  [poetry] 26 files found in ./corpi_these/corpus_poesie_hetero
  [prose] 55 files found in ./corpi_these/corpus_hetero
Total chunks : 11961
genre
prose     10279
poetry     1682
Name: count, dtype: int64


Batches: 100%|██████████| 94/94 [00:11<00:00,  7.86it/s]
2026-04-27 13:09:49,799 - BERTopic - WARNING: Note that extracting more than 100 words from a sparse can slow down computation quite a bit.
2026-04-27 13:09:49,836 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-27 13:09:54,311 - BERTopic - Dimensionality - Completed ✓
2026-04-27 13:09:54,313 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-27 13:09:54,997 - BERTopic - Cluster - Completed ✓
2026-04-27 13:09:55,002 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-27 13:10:00,287 - BERTopic - Representation - Completed ✓


  Outliers before reduction : 4735 (39.6%)
  After reduce_outliers (embeddings): 0 remaining outliers
  Total outliers reduced: 4735 (100.0% reassigned)
Topics (excl. -1) : 305
Proportion outliers (topic=-1) : 0.00%
[f_hetero] Main loop results saved in /home/vsaiag/Desktop/article1_test/final_results/f_hetero
Analysis of corpus homo_m
  [poetry] 33 files found in ./corpi_these/corpus_poesie_homosexuels
  [prose] 64 files found in ./corpi_these/corpus_homosexuels
Total chunks : 13555
genre
prose     11210
poetry     2345
Name: count, dtype: int64


Batches: 100%|██████████| 106/106 [00:13<00:00,  7.87it/s]
2026-04-27 13:14:56,405 - BERTopic - WARNING: Note that extracting more than 100 words from a sparse can slow down computation quite a bit.
2026-04-27 13:14:56,432 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-27 13:15:01,759 - BERTopic - Dimensionality - Completed ✓
2026-04-27 13:15:01,761 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-27 13:15:02,508 - BERTopic - Cluster - Completed ✓
2026-04-27 13:15:02,513 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-27 13:15:08,804 - BERTopic - Representation - Completed ✓


  Outliers before reduction : 5339 (39.4%)
  After reduce_outliers (embeddings): 0 remaining outliers
  Total outliers reduced: 5339 (100.0% reassigned)
Topics (excl. -1) : 313
Proportion outliers (topic=-1) : 0.00%
[homo_m] Main loop results saved in /home/vsaiag/Desktop/article1_test/final_results/homo_m
Analysis of corpus h_hetero
  [poetry] 34 files found in ./corpi_these/corpus_poesie_homhet
  [prose] 140 files found in ./corpi_these/corpus_homhet
Total chunks : 31929
genre
prose     30527
poetry     1402
Name: count, dtype: int64


Batches: 100%|██████████| 250/250 [00:32<00:00,  7.62it/s]
2026-04-27 13:22:29,515 - BERTopic - WARNING: Note that extracting more than 100 words from a sparse can slow down computation quite a bit.
2026-04-27 13:22:29,548 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-27 13:22:42,842 - BERTopic - Dimensionality - Completed ✓
2026-04-27 13:22:42,843 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-27 13:22:46,823 - BERTopic - Cluster - Completed ✓
2026-04-27 13:22:46,834 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-27 13:23:03,145 - BERTopic - Representation - Completed ✓


  Outliers before reduction : 15632 (49.0%)
  After reduce_outliers (embeddings): 0 remaining outliers
  Total outliers reduced: 15632 (100.0% reassigned)
Topics (excl. -1) : 681
Proportion outliers (topic=-1) : 0.00%
[h_hetero] Main loop results saved in /home/vsaiag/Desktop/article1_test/final_results/h_hetero

 All corpora have been analyzed for Q1


In [10]:
# 9b - Exporting comparative synthesis for Q1
# No additional analyses were performed here. This cell only helps us organise our results

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

SYNTHESE_ROUND   = 4
SYNTHESE_RUN     = RUN_ID_REF
SYNTHESE_MODE    = MODE_REF
SYNTHESE_OUT_DIR = os.path.join(OUT_DIR_ROOT, '_synthese_comparative')
os.makedirs(SYNTHESE_OUT_DIR, exist_ok=True)

COLOR_HIGH   = 'C6EFCE'
COLOR_HEADER = '4472C4'
COLOR_SUBHDR = 'D9E1F2'
COLOR_ALT    = 'F2F2F2'

def _border():
    thin = Side(style='thin', color='BFBFBF')
    return Border(left=thin, right=thin, top=thin, bottom=thin)

def _header_cell(ws, row, col, value, bg=COLOR_HEADER, font_color='FFFFFF', bold=True):
    c = ws.cell(row=row, column=col, value=value)
    c.fill      = PatternFill('solid', fgColor=bg)
    c.font      = Font(bold=bold, color=font_color)
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border    = _border()
    return c

def _data_cell(ws, row, col, value, bold=False, bg=None):
    c = ws.cell(row=row, column=col, value=value)
    c.font      = Font(bold=bold)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border    = _border()
    if bg:
        c.fill = PatternFill('solid', fgColor=bg)
    return c

def _interpolate_green(ratio):
    """
    ratio between 0 et 1.
    0   → white (FFFFFF)
    1   → green (1E7B34)
    """
    r = int(255 - ratio * (255 - 30))
    g = int(255 - ratio * (255 - 123))
    b = int(255 - ratio * (255 - 52))
    return f'{r:02X}{g:02X}{b:02X}'

def _set_col_widths(ws, widths):
    for col, width in widths.items():
        ws.column_dimensions[get_column_letter(col)].width = width

def load_work_scores_all_corpora():
    result = {}
    TOPN_REF = RUNS_TOPN[SYNTHESE_RUN]
    for cfg in CORPUS_CONFIGS:
        name     = cfg['name']
        csv_path = os.path.join(
            OUT_DIR_ROOT, name,
            f'mode_{SYNTHESE_MODE}',
            f'run_{SYNTHESE_RUN}_topn_{TOPN_REF}',
            'work_level_scores.csv')
        if not os.path.isfile(csv_path):
            print(f'  [Synthèse] Fichier manquant, skip : {csv_path}')
            continue
        result[name] = pd.read_csv(csv_path)
    return result

def compute_means(work_scores_dict, lex_name, score_prefix):
    col  = f'{score_prefix}{lex_name}'
    rows = []
    for corpus_name, df in work_scores_dict.items():
        if col not in df.columns:
            continue
        mp = df.loc[df['genre'] == 'poetry', col].mean()
        mr = df.loc[df['genre'] == 'prose',  col].mean()
        rows.append({
            'corpus':              corpus_name,
            'mean_poetry':         round(float(mp), SYNTHESE_ROUND) if not pd.isna(mp) else None,
            'mean_prose':          round(float(mr), SYNTHESE_ROUND) if not pd.isna(mr) else None,
            'diff_poetry_minus_prose': round(float(mp - mr), SYNTHESE_ROUND)
                                   if not (pd.isna(mp) or pd.isna(mr)) else None,
        })
    return pd.DataFrame(rows)

def write_comparison_sheet(ws, df_means, lex_name, score_type_label):
    ws.freeze_panes = 'B3'
    ws.merge_cells('A1:D1')
    t = ws['A1']
    t.value     = f'Lexique : {lex_name}  |  Score : {score_type_label}'
    t.font      = Font(bold=True, size=13, color='FFFFFF')
    t.fill      = PatternFill('solid', fgColor=COLOR_HEADER)
    t.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 22
    for ci, h in enumerate(['Corpus', 'mean_poetry', 'mean_prose', 'diff (poetry - prose)'], start=1):
        _header_cell(ws, 2, ci, h, bg=COLOR_SUBHDR, font_color='000000')
    ws.row_dimensions[2].height = 18


    all_vals = []
    for _, row in df_means.iterrows():
        if pd.notna(row['mean_poetry']):
            all_vals.append(row['mean_poetry'])
        if pd.notna(row['mean_prose']):
            all_vals.append(row['mean_prose'])
    global_min = min(all_vals) if all_vals else 0
    global_max = max(all_vals) if all_vals else 1
    global_range = global_max - global_min if global_max != global_min else 1

    for ri, (_, row) in enumerate(df_means.iterrows(), start=3):
        bg_row = COLOR_ALT if ri % 2 == 0 else None
        _data_cell(ws, ri, 1, row['corpus'], bg=bg_row)

        mp  = row['mean_poetry']
        mr  = row['mean_prose']
        dif = row['diff_poetry_minus_prose']

        bold_p = (pd.notna(mp) and pd.notna(mr) and mp > mr)
        bold_r = (pd.notna(mp) and pd.notna(mr) and mr > mp)

        if pd.notna(mp):
            ratio_p = (mp - global_min) / global_range
            bg_p    = _interpolate_green(ratio_p)
        else:
            bg_p = bg_row

        if pd.notna(mr):
            ratio_r = (mr - global_min) / global_range
            bg_r    = _interpolate_green(ratio_r)
        else:
            bg_r = bg_row

        _data_cell(ws, ri, 2, mp, bold=bold_p, bg=bg_p)
        _data_cell(ws, ri, 3, mr, bold=bold_r, bg=bg_r)
        _data_cell(ws, ri, 4, dif, bg=bg_row)

    _set_col_widths(ws, {1: 18, 2: 16, 3: 16, 4: 22})

def export_per_lexicon(work_scores_dict, lex_names):
    for lex in lex_names:
        wb = openpyxl.Workbook()
        wb.remove(wb.active)
        for prefix, label in [('score_', 'Topic-affinity'),
                               ('score_direct_', 'Score direct lématique')]:
            df_means = compute_means(work_scores_dict, lex, prefix)
            if df_means.empty:
                continue
            ws = wb.create_sheet(title=label[:31])
            write_comparison_sheet(ws, df_means, lex, label)
        out_path = os.path.join(SYNTHESE_OUT_DIR, f'synthese_Q1_{lex}.xlsx')
        wb.save(out_path)
        print(f'  [Synthesis] Exported to : {out_path}')

def export_all_lexicons(work_scores_dict, lex_names):
    wb = openpyxl.Workbook()
    wb.remove(wb.active)
    for lex in lex_names:
        for prefix, label in [('score_direct_', 'Direct'), ('score_', 'Affinity')]:
            df_means = compute_means(work_scores_dict, lex, prefix)
            if df_means.empty:
                continue
            ws = wb.create_sheet(title=f'{lex[:18]}_{label}'[:31])
            write_comparison_sheet(ws, df_means, lex, label)

    ws_g = wb.create_sheet(title='Vue globale', index=0)
    ws_g.freeze_panes = 'C3'
    corpus_names = list(work_scores_dict.keys())
    col_defs = []
    for lex in lex_names:
        for prefix, lbl in [('score_direct_', 'Dir.'), ('score_', 'Aff.')]:
            col_defs.append((lex, prefix, lbl, 'poetry'))
            col_defs.append((lex, prefix, lbl, 'prose'))
    n = len(col_defs)
    ws_g.merge_cells(start_row=1, start_column=1, end_row=1, end_column=1 + n)
    t = ws_g.cell(row=1, column=1, value='Vue globale — Q1 · Tous lexiques · Tous corpus')
    t.font      = Font(bold=True, size=13, color='FFFFFF')
    t.fill      = PatternFill('solid', fgColor=COLOR_HEADER)
    t.alignment = Alignment(horizontal='center', vertical='center')
    ws_g.row_dimensions[1].height = 22
    _header_cell(ws_g, 2, 1, 'Corpus', bg=COLOR_SUBHDR, font_color='000000')
    for ci, (lex, prefix, lbl, genre) in enumerate(col_defs, start=2):
        _header_cell(ws_g, 2, ci, f'{lex}\n{lbl}\n{genre}',
                     bg=COLOR_SUBHDR, font_color='000000')
    ws_g.row_dimensions[2].height = 48

    means_cache = {}
    for lex in lex_names:
        for prefix, lbl in [('score_direct_', 'Dir.'), ('score_', 'Aff.')]:
            col = f'{prefix}{lex}'
            for corpus_name, df in work_scores_dict.items():
                if col not in df.columns:
                    continue
                for genre in ('poetry', 'prose'):
                    v = df.loc[df['genre'] == genre, col].mean()
                    means_cache[(lex, prefix, corpus_name, genre)] = (
                        round(float(v), SYNTHESE_ROUND) if not pd.isna(v) else None)

    for ri, corpus_name in enumerate(corpus_names, start=3):
        bg_row = COLOR_ALT if ri % 2 == 0 else None
        _data_cell(ws_g, ri, 1, corpus_name, bold=True, bg=bg_row)
        for ci, (lex, prefix, lbl, genre) in enumerate(col_defs, start=2):
            v        = means_cache.get((lex, prefix, corpus_name, genre))
            v_poetry = means_cache.get((lex, prefix, corpus_name, 'poetry'))
            v_prose  = means_cache.get((lex, prefix, corpus_name, 'prose'))
            is_high  = (v is not None and v_poetry is not None and v_prose is not None
                        and v == max(v_poetry, v_prose) and v_poetry != v_prose)
            _data_cell(ws_g, ri, ci, v, bold=is_high,
                       bg=COLOR_HIGH if is_high else bg_row)
    ws_g.column_dimensions['A'].width = 16
    for ci in range(2, 2 + n):
        ws_g.column_dimensions[get_column_letter(ci)].width = 13

    out_path = os.path.join(SYNTHESE_OUT_DIR, 'synthese_Q1_tous_lexiques.xlsx')
    wb.save(out_path)
    print(f'  [Synthesis] Exported to: {out_path}')

print(f'\nLoading work_level_scores (run={SYNTHESE_RUN}, mode={SYNTHESE_MODE})...')
_work_scores_dict = load_work_scores_all_corpora()
print(f'Corpora loaded : {list(_work_scores_dict.keys())}')
export_per_lexicon(_work_scores_dict, lexicon_names)
export_all_lexicons(_work_scores_dict, lexicon_names)
print(f'\n Results exported in : {os.path.abspath(SYNTHESE_OUT_DIR)}')


Loading work_level_scores (run=B, mode=lemma_only)...
Corpora loaded : ['lesbiennes', 'f_hetero', 'homo_m', 'h_hetero']
  [Synthesis] Exported to : ./final_results/_synthese_comparative/synthese_Q1_lesbian_love.xlsx
  [Synthesis] Exported to : ./final_results/_synthese_comparative/synthese_Q1_love.xlsx
  [Synthesis] Exported to : ./final_results/_synthese_comparative/synthese_Q1_love_and_desire.xlsx
  [Synthesis] Exported to : ./final_results/_synthese_comparative/synthese_Q1_physical_desire.xlsx
  [Synthesis] Exported to : ./final_results/_synthese_comparative/synthese_Q1_sapphism.xlsx
  [Synthesis] Exported to: ./final_results/_synthese_comparative/synthese_Q1_tous_lexiques.xlsx

 Results exported in : /home/vsaiag/Desktop/article1_test/final_results/_synthese_comparative


In [11]:
# 10 - Q2: Differential analysis (lesbian corpus)

# Q2 asks whether lesbian authors spoke differently about lesbianism in poetry and in novel

# For each lexicon, chunks with a direct score > 0 are analysed along four axes:


# 1) Topic distribution: Chi-squared test of independence and
# Jensen-Shannon divergence between poetic and narrative topic distributions. 

def js_divergence(p, q) -> float:
    p, q = np.array(p, dtype=float), np.array(q, dtype=float)
    p = p / p.sum() if p.sum() > 0 else p
    q = q / q.sum() if q.sum() > 0 else q
    return float(jensenshannon(p, q, base=2))


def axe1_topic_distribution(df_filtered, valid_topic_ids_local, out_dir, label,
                             n_total_poetry=None, n_total_prose=None):
    poetry = df_filtered[df_filtered['genre'] == 'poetry']
    prose  = df_filtered[df_filtered['genre'] == 'prose']
    if poetry.empty or prose.empty:
        print(f'  [Axis 1 | {label}] Not enough chunks, skip.')
        return

    all_topics  = [t for t in valid_topic_ids_local if t != -1]
    p_counts    = poetry['topic'].value_counts().reindex(all_topics, fill_value=0)
    r_counts    = prose['topic'].value_counts().reindex(all_topics, fill_value=0)
    contingency = np.array([p_counts.values, r_counts.values])
    nonzero     = contingency.sum(axis=0) > 0
    contingency = contingency[:, nonzero]
    topics_kept = np.array(all_topics)[nonzero]

    chi2_val, p_val, dof = np.nan, np.nan, np.nan
    row_sums = contingency.sum(axis=1)
    if contingency.shape[1] >= 2 and np.all(row_sums > 0):
        try:
            chi2_val, p_val, dof, _ = chi2_contingency(contingency)
        except ValueError as e:
            print(f'  [Axis 1 | {label}] Chi² impossible : {e}')

    js_div = js_divergence(p_counts.values, r_counts.values)

    # Normalisation 
    denom_p = n_total_poetry if n_total_poetry else max(contingency[0].sum(), 1)
    denom_r = n_total_prose  if n_total_prose  else max(contingency[1].sum(), 1)

    share_p = contingency[0] / denom_p
    share_r = contingency[1] / denom_r

    topic_summary = pd.DataFrame({
        'topic':                topics_kept,
        'n_poetry':             contingency[0],
        'n_prose':              contingency[1],
        'share_poetry_in_corpus': share_p,
        'share_prose_in_corpus':  share_r,
    })

    topic_summary['ratio_poetry_over_prose'] = (
        share_p / np.where(share_r == 0, np.nan, share_r))
    topic_summary = topic_summary.sort_values('ratio_poetry_over_prose', ascending=False)

    global_stats = pd.DataFrame([{
        'lexicon':           label,
        'n_chunks_poetry':   len(poetry),
        'n_chunks_prose':    len(prose),
        'n_total_poetry':    n_total_poetry,
        'n_total_prose':     n_total_prose,
        'share_corpus_poetry_retained': len(poetry) / denom_p,
        'share_corpus_prose_retained':  len(prose)  / denom_r,
        'chi2':              chi2_val,
        'p_value':           p_val,
        'dof':               dof,
        'js_divergence':     js_div,
        'n_topics_compared': int(nonzero.sum())
    }])

    topic_summary.to_csv(
        os.path.join(out_dir, f'axe1_topic_distribution_{label}.csv'), index=False)
    global_stats.to_csv(
        os.path.join(out_dir, f'axe1_global_stats_{label}.csv'), index=False)
    print(f'  [Axis 1 | {label}] Chi²={chi2_val:.3f}, p={p_val:.4f}, JS={js_div:.4f}')
    print(f'  [Axis 1 | {label}] Part of the corpus retained : '
          f'poetry={len(poetry)/denom_p:.2%}, prose={len(prose)/denom_r:.2%}')


# 2) Differential TF-IDF vocabulary: most characteristic words of
# each genre among filtered chunks, ranked by difference in mean
# TF-IDF score between poetry and prose

def axe2_tfidf_differential(df_filtered, out_dir, label, top_n=TFIDF_TOP_N):
    """The most characteristic words for each genre in the filtered chunks
    (differential TF-IDF using text_vectorizer, excluding function words)."""
    poetry_texts = df_filtered[df_filtered['genre'] == 'poetry']['text_vectorizer'].tolist()
    prose_texts  = df_filtered[df_filtered['genre'] == 'prose']['text_vectorizer'].tolist()
    if len(poetry_texts) < 2 or len(prose_texts) < 2:
        print(f'  [Axis 2 | {label}] Not enough documents, skip.')
        return

    all_texts = poetry_texts + prose_texts
    genre_vec  = ['poetry'] * len(poetry_texts) + ['prose'] * len(prose_texts)

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95,
        stop_words=list(STOPWORDS_SINGLE)
    )
    tfidf_mat  = vectorizer.fit_transform(all_texts)
    features   = np.array(vectorizer.get_feature_names_out())

    p_idx  = [i for i, g in enumerate(genre_vec) if g == 'poetry']
    r_idx  = [i for i, g in enumerate(genre_vec) if g == 'prose']
    p_mean = np.asarray(tfidf_mat[p_idx].mean(axis=0)).flatten()
    r_mean = np.asarray(tfidf_mat[r_idx].mean(axis=0)).flatten()
    diff   = p_mean - r_mean

    df_tfidf = (
        pd.DataFrame({'word': features, 'tfidf_poetry': p_mean,
                      'tfidf_prose': r_mean, 'diff_poetry_minus_prose': diff})
        .sort_values('diff_poetry_minus_prose', ascending=False)
    )

    top_p = df_tfidf.head(top_n).assign(characteristic_of='poetry')
    top_r = (df_tfidf.tail(top_n)
             .sort_values('diff_poetry_minus_prose', ascending=True)
             .assign(characteristic_of='prose'))
    result = pd.concat([top_p, top_r], ignore_index=True)
    result.to_csv(os.path.join(out_dir, f'axe2_tfidf_differential_{label}.csv'), index=False)
    print(f'  [Axis 2 | {label}] Top poetry : {list(top_p["word"].head(5))}')
    print(f'  [Axis 2 | {label}] Top prose  : {list(top_r["word"].head(5))}')


# 3) Contextual co-occurrences: words appearing within a COOC_WINDOW
# around each lexicon word occurrence, compared across genres via 
# differential TF-IDF and a Chi-squared test on the top co-occurrence words

def extract_context_windows(tokens, lex_entries, window, stopwords, lemma_map=None):
    context_words = []
    for i, tok in enumerate(tokens):
        surf = norm_token(tok)
        lem  = lemma_map.get(surf, surf) if lemma_map else surf
        if lem not in lex_entries:
            continue
        left  = max(0, i - window)
        right = min(len(tokens), i + window + 1)
        for j in range(left, right):
            if j == i:
                continue
            w = norm_token(tokens[j])
            if w and w not in stopwords and len(w) > 1 and not w.isdigit():
                context_words.append(w)
    return context_words


def axe3_cooccurrence(df_filtered, lex_entries, out_dir, label,
                      window=COOC_WINDOW, top_n=TFIDF_TOP_N, lemma_map=None):
    """
    Compares the contextual words surrounding lexical entries
    between poetry and prose.

    Output files:
      axe3_cooc_tfidf_{label}.csv   — characteristic words by genre
      axe3_cooc_chi2_{label}.csv    — Chi-square test on shared top words
    """
    poetry_df = df_filtered[df_filtered['genre'] == 'poetry']
    prose_df  = df_filtered[df_filtered['genre'] == 'prose']

    if poetry_df.empty or prose_df.empty:
        print(f'  [Axis 3 | {label}] Not enough chunks, skip.')
        return

    def collect_windows(sub_df):
        """Aggregates the context windows of all chunks of a given type.
        Returns a list of strings (one per chunk) for TF-IDF,
        and a global Counter for Chi²."""
        docs, global_counter = [], Counter()
        for raw in sub_df['text_raw']:
            toks = tokenize(raw)
            ctx = extract_context_windows(toks, lex_entries, window, STOPWORDS_SINGLE,
                               lemma_map=lemma_map)
            if ctx:
                docs.append(' '.join(ctx))
                global_counter.update(ctx)
        return docs, global_counter

    poetry_docs, poetry_counter = collect_windows(poetry_df)
    prose_docs,  prose_counter  = collect_windows(prose_df)

    print(f'  [Axis 3 | {label}] Lexical occurrences: '
          f'poetry={sum(poetry_counter.values())}, '
          f'prose={sum(prose_counter.values())}')

    if len(poetry_docs) >= 2 and len(prose_docs) >= 2:
        all_docs  = poetry_docs + prose_docs
        genre_vec = ['poetry'] * len(poetry_docs) + ['prose'] * len(prose_docs)

        vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 1), min_df=2, max_df=0.95,
                      stop_words=list(STOPWORDS_SINGLE))
        mat = vec.fit_transform(all_docs)
        feats = np.array(vec.get_feature_names_out())

        p_idx  = [i for i, g in enumerate(genre_vec) if g == 'poetry']
        r_idx  = [i for i, g in enumerate(genre_vec) if g == 'prose']
        p_mean = np.asarray(mat[p_idx].mean(axis=0)).flatten()
        r_mean = np.asarray(mat[r_idx].mean(axis=0)).flatten()
        diff   = p_mean - r_mean

        df_cooc_tfidf = (
            pd.DataFrame({'word': feats, 'tfidf_poetry': p_mean,
                          'tfidf_prose': r_mean, 'diff_poetry_minus_prose': diff})
            .sort_values('diff_poetry_minus_prose', ascending=False)
        )
        top_p = df_cooc_tfidf.head(top_n).assign(characteristic_of='poetry')
        top_r = (df_cooc_tfidf.tail(top_n)
                 .sort_values('diff_poetry_minus_prose', ascending=True)
                 .assign(characteristic_of='prose'))
        result_tfidf = pd.concat([top_p, top_r], ignore_index=True)
        result_tfidf.to_csv(
            os.path.join(out_dir, f'axe3_cooc_tfidf_{label}.csv'), index=False)
        print(f'  [Axis 3 | {label}] Top cooc poetry : {list(top_p["word"].head(5))}')
        print(f'  [Axis 3 | {label}] Top cooc prose  : {list(top_r["word"].head(5))}')
    else:
        print(f'  [Axis 3 | {label}] Not enough documents for TF-IDF, skip TF-IDF.')

    all_words_sorted = sorted(
        (poetry_counter + prose_counter).keys(),
        key=lambda w: -(poetry_counter[w] + prose_counter[w])
    )[:top_n * 2]

    if len(all_words_sorted) >= 2:
        contingency = np.array([
            [poetry_counter.get(w, 0) for w in all_words_sorted],
            [prose_counter.get(w, 0)  for w in all_words_sorted]
        ])
        nonzero = contingency.sum(axis=0) > 0
        cont_nz = contingency[:, nonzero]
        words_nz = np.array(all_words_sorted)[nonzero]

        chi2_val, p_val, dof = np.nan, np.nan, np.nan
        row_sums = contingency.sum(axis=1)
        if contingency.shape[1] >= 2 and np.all(row_sums > 0):
            try:
                chi2_val, p_val, dof, _ = chi2_contingency(contingency)
            except ValueError as e:
                print(f'  [Axis 3 | {label}] Chi² impossible : {e}')

        total_p = max(sum(poetry_counter.values()), 1)
        total_r = max(sum(prose_counter.values()), 1)

        df_chi2_words = pd.DataFrame({
            'word':         words_nz,
            'n_poetry':     cont_nz[0],
            'n_prose':      cont_nz[1],
            'freq_poetry':  cont_nz[0] / total_p,
            'freq_prose':   cont_nz[1] / total_r,
            'total':        cont_nz[0] + cont_nz[1],
            'share_poetry': cont_nz[0] / np.maximum(cont_nz[0] + cont_nz[1], 1),
        }).sort_values('total', ascending=False)

        global_chi2 = pd.DataFrame([{
            'lexicon': label,
            'chi2': chi2_val, 'p_value': p_val, 'dof': dof,
            'n_words_tested': int(nonzero.sum()),
            'n_cooc_poetry': sum(poetry_counter.values()),
            'n_cooc_prose':  sum(prose_counter.values()),
        }])
        df_chi2_words.to_csv(
            os.path.join(out_dir, f'axe3_cooc_chi2_words_{label}.csv'), index=False)
        global_chi2.to_csv(
            os.path.join(out_dir, f'axe3_cooc_chi2_global_{label}.csv'), index=False)
        print(f'  [Axis 3 | {label}] Global Chi²={chi2_val:.3f}, p={p_val:.4f}')
    else:
        print(f'  [Axis 3 | {label}] Not enough shared words for Chi², skip.')


# 4) Emotional valence: CamemBERT-based sentiment scores (1 to 5) compared 
# across genres via Mann-Whitney U test and Cliff's delta

def axe4_sentiment(df_filtered, out_dir, label):
    """CamemBERT scores on the filtered chunks.
    The model returns a score of 1 to 5 stars (1 = very negative, 5 = very positive)."""
    if df_filtered.empty:
        print(f'  [Axis 4 | {label}] Empty DataFrame, skip.')
        return

    texts = df_filtered['text_raw'].str[:SENTIMENT_MAX_CHARS].tolist()
    scores_num, labels_str = [], []
    unexpected_labels = []

    print(f'  [Axis 4 | {label}] Sentiment analysis on {len(texts)} chunks...')
    for i in range(0, len(texts), SENTIMENT_BATCH_SIZE):
        batch = texts[i:i + SENTIMENT_BATCH_SIZE]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            results = sentiment_pipeline(batch)
        for r in results:
            try:
                star_num = int(r['label'].split()[0])
            except (ValueError, IndexError):
                unexpected_labels.append(r['label'])  
                star_num = 3
            scores_num.append(star_num)
            labels_str.append(r['label'])

    if unexpected_labels:
        print(f'  [Axis 4 | {label}] Unexpected labels (fallback=3) : '
              f'{Counter(unexpected_labels)}') 

    df_sent = df_filtered[['chunk_id', 'work_id', 'author_id', 'genre']].copy()
    df_sent['sentiment_score'] = scores_num
    df_sent['sentiment_label'] = labels_str

    x = df_sent[df_sent['genre'] == 'poetry']['sentiment_score'].to_numpy()
    y = df_sent[df_sent['genre'] == 'prose']['sentiment_score'].to_numpy()
    U, p, d = np.nan, np.nan, np.nan
    if len(x) > 0 and len(y) > 0:
        U, p = mannwhitneyu(x, y, alternative='two-sided')
        d    = cliffs_delta(x, y)

    global_stats = pd.DataFrame([{
        'lexicon': label, 'n_poetry': len(x), 'n_prose': len(y),
        'mean_sentiment_poetry': float(np.mean(x)) if len(x) > 0 else np.nan,
        'mean_sentiment_prose':  float(np.mean(y)) if len(y) > 0 else np.nan,
        'MWU_U': U, 'p_value': p, 'cliffs_delta': d
    }])

    label_dist = (
        df_sent.groupby(['genre', 'sentiment_label']).size()
        .reset_index(name='n_chunks')
    )
    label_dist['share'] = (
        label_dist.groupby('genre')['n_chunks'].transform(lambda s: s / s.sum()))

    df_sent.to_csv(
        os.path.join(out_dir, f'axe4_sentiment_chunks_{label}.csv'), index=False)
    global_stats.to_csv(
        os.path.join(out_dir, f'axe4_sentiment_stats_{label}.csv'), index=False)
    label_dist.to_csv(
        os.path.join(out_dir, f'axe4_sentiment_distribution_{label}.csv'), index=False)
    print(f'  [Axis 4 | {label}] Average for poetry={np.nanmean(x):.2f}, '
          f'prose={np.nanmean(y):.2f}, p={p:.4f}, d={d:.3f}')



for corpus_cfg in [c for c in CORPUS_CONFIGS if c['name'] == 'lesbiennes']:
    corpus_name = corpus_cfg['name']
    OUT_DIR     = os.path.join(OUT_DIR_ROOT, corpus_name)
    TOPN_REF    = RUNS_TOPN[RUN_ID_REF]
    ref_run_dir = os.path.join(
        OUT_DIR, f'mode_{MODE_REF}', f'run_{RUN_ID_REF}_topn_{TOPN_REF}')
    diff_dir    = os.path.join(OUT_DIR, 'analyse_differentielle')
    os.makedirs(diff_dir, exist_ok=True)

    print(f"Analysis of corpus {corpus_name}")

    csv_path = os.path.join(ref_run_dir, 'chunk_level_scores.csv')
    if not os.path.isfile(csv_path):
        print(f'  [WARN] chunk_level_scores.csv could not be found in {ref_run_dir}, skip.')
        continue

    df_scored = pd.read_csv(csv_path)
    required  = {'chunk_id', 'work_id', 'author_id', 'genre', 'text_raw', 'text_vectorizer', 'topic'}
    missing   = required - set(df_scored.columns)
    if missing:
        print(f'  [WARN] Missing columns: {missing}. Skip.')
        continue

    valid_topic_ids_local = [
        int(t) for t in df_scored['topic'].dropna().unique().tolist()
        if int(t) != -1
    ]

    for lex_label in TARGET_LEXICONS:
        score_direct_col = f'score_direct_{lex_label}'
        if score_direct_col not in df_scored.columns:
            print(f'  [WARN] Column "{score_direct_col}" missing. '
                  f'Rerun cell 9. Skip "{lex_label}".')
            continue

        df_lex = df_scored[df_scored[score_direct_col] > SCORE_THRESHOLD_DIRECT].copy()
        n_p = len(df_lex[df_lex['genre'] == 'poetry'])
        n_r = len(df_lex[df_lex['genre'] == 'prose'])
        print(f'\n--- Lexicon : {lex_label} | retained chunks: {len(df_lex)} '
              f'(poetry={n_p}, prose={n_r}) ---')

        if df_lex.empty:
            print(f'  [WARN] No chunks retained for "{lex_label}", skip.')
            continue

        lex_out_dir = os.path.join(diff_dir, lex_label)
        os.makedirs(lex_out_dir, exist_ok=True)

        lex_entries = LEXICON_MAP[lex_label]

        n_total_poetry = int((df_scored['genre'] == 'poetry').sum())
        n_total_prose  = int((df_scored['genre'] == 'prose').sum())
        axe1_topic_distribution(df_lex, valid_topic_ids_local, lex_out_dir, lex_label,
                        n_total_poetry=n_total_poetry, n_total_prose=n_total_prose)
        axe2_tfidf_differential(df_lex, lex_out_dir, lex_label)
        axe3_cooccurrence(df_lex, lex_entries, lex_out_dir, lex_label,
                  lemma_map=surface_best_lemma)
        axe4_sentiment(df_lex, lex_out_dir, lex_label)     
        
    print(f' Results saved in {os.path.abspath(diff_dir)}')

print('\n Differential analysis Q2 done.')

Analysis of corpus lesbiennes

--- Lexicon : lesbian_love | retained chunks: 7589 (poetry=912, prose=6677) ---
  [Axis 1 | lesbian_love] Chi²=2690.784, p=0.0000, JS=0.7026
  [Axis 1 | lesbian_love] Part of the corpus retained : poetry=74.69%, prose=45.71%
  [Axis 2 | lesbian_love] Top poetry : ['ame', 'mer', 'ombre', 'soir', 'mort']
  [Axis 2 | lesbian_love] Top prose  : ['petite', 'faire', 'encore', 'homme', 'stephen']
  [Axis 3 | lesbian_love] Lexical occurrences: poetry=14175, prose=69097
  [Axis 3 | lesbian_love] Top cooc poetry : ['ame', 'cœur', 'lys', 'mer', 'mort']
  [Axis 3 | lesbian_love] Top cooc prose  : ['homme', 'petite', 'femme', 'faire', 'vie']
  [Axis 3 | lesbian_love] Global Chi²=717.008, p=0.0000
  [Axis 4 | lesbian_love] Sentiment analysis on 7589 chunks...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [Axis 4 | lesbian_love] Average for poetry=4.88, prose=3.69, p=0.0000, d=0.396

--- Lexicon : love | retained chunks: 7532 (poetry=895, prose=6637) ---
  [Axis 1 | love] Chi²=2663.199, p=0.0000, JS=0.7043
  [Axis 1 | love] Part of the corpus retained : poetry=73.30%, prose=45.44%
  [Axis 2 | love] Top poetry : ['ame', 'mer', 'ombre', 'soir', 'mort']
  [Axis 2 | love] Top prose  : ['petite', 'faire', 'encore', 'homme', 'stephen']
  [Axis 3 | love] Lexical occurrences: poetry=11380, prose=66061
  [Axis 3 | love] Top cooc poetry : ['ame', 'cœur', 'lys', 'ombre', 'mer']
  [Axis 3 | love] Top cooc prose  : ['homme', 'femme', 'petite', 'vie', 'faire']
  [Axis 3 | love] Global Chi²=623.715, p=0.0000
  [Axis 4 | love] Sentiment analysis on 7532 chunks...
  [Axis 4 | love] Average for poetry=4.88, prose=3.68, p=0.0000, d=0.396

--- Lexicon : love_and_desire | retained chunks: 12667 (poetry=1126, prose=11541) ---
  [Axis 1 | love_and_desire] Chi²=4056.208, p=0.0000, JS=0.7098
  [Axis 1 | love_

In [12]:
# 11 - Synthesis for Q2

# No additional analyses were performed here. This cell only helps us organise our results

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

SYNTHESE_Q2_TOPN_WORDS = 10   
SYNTHESE_Q2_TOP_WORDS  = 5    

# Colours
_C_HEADER  = '4472C4'
_C_SUBHDR  = 'D9E1F2'
_C_ALT     = 'F2F2F2'
_C_POETRY  = 'E2EFDA'   # pale green - poetry line
_C_PROSE   = 'DDEBF7'   # light blue - prose line
_C_SECTION = 'FFF2CC'   # pale yellow - section title
_C_SIG     = 'C6EFCE'   # green - significant result
_C_NONSIG  = 'FFEB9C'   # light orange - non-significant result

def _bdr():
    t = Side(style='thin', color='BFBFBF')
    return Border(left=t, right=t, top=t, bottom=t)

def _hcell(ws, r, c, v, bg=_C_HEADER, fc='FFFFFF', bold=True, size=11):
    cell = ws.cell(row=r, column=c, value=v)
    cell.fill      = PatternFill('solid', fgColor=bg)
    cell.font      = Font(bold=bold, color=fc, size=size)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border    = _bdr()
    return cell

def _dcell(ws, r, c, v, bold=False, bg=None, italic=False, wrap=False):
    cell = ws.cell(row=r, column=c, value=v)
    cell.font      = Font(bold=bold, italic=italic)
    cell.alignment = Alignment(horizontal='left', vertical='center', wrap_text=wrap)
    cell.border    = _bdr()
    if bg:
        cell.fill = PatternFill('solid', fgColor=bg)
    return cell

def _section_title(ws, r, c_start, c_end, title):
    ws.merge_cells(start_row=r, start_column=c_start,
                   end_row=r,   end_column=c_end)
    cell = ws.cell(row=r, column=c_start, value=title)
    cell.fill      = PatternFill('solid', fgColor=_C_SECTION)
    cell.font      = Font(bold=True, size=11)
    cell.alignment = Alignment(horizontal='left', vertical='center')
    cell.border    = _bdr()
    ws.row_dimensions[r].height = 16


def _read_csv_safe(path):
    """Reads a CSV file if it exists, or else returns None."""
    if os.path.isfile(path):
        try:
            return pd.read_csv(path)
        except Exception:
            return None
    return None

def _get_topic_keywords(topic_words_df, topic_id, n=SYNTHESE_Q2_TOPN_WORDS):
    """Returns the n keywords of a topic in a string format"""
    if topic_words_df is None:
        return '—'
    sub = topic_words_df[topic_words_df['topic'] == topic_id].sort_values('rank')
    words = sub['word'].head(n).tolist()
    return ', '.join(words) if words else '—'

def build_q2_summary_for_corpus(corpus_name):
    """
    Builds the file synthese_Q2_{corpus}.xlsx.
    Returns the file's path or None.
    """
    corpus_out_dir  = os.path.join(OUT_DIR_ROOT, corpus_name)
    diff_dir        = os.path.join(corpus_out_dir, 'analyse_differentielle')
    topic_words_path = os.path.join(corpus_out_dir, 'bertopic_topic_words_with_lemma.csv')

    if not os.path.isdir(diff_dir):
        print(f'  [Synthesis Q2 | {corpus_name}] File analyse_differentielle absent, skip.')
        return None

    topic_words_df = _read_csv_safe(topic_words_path)
    if topic_words_df is None:
        print(f'  [Synthesis Q2 | {corpus_name}] bertopic_topic_words_with_lemma.csv absent.')

    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    global_rows = []

    for lex_label in TARGET_LEXICONS:
        lex_dir = os.path.join(diff_dir, lex_label)
        if not os.path.isdir(lex_dir):
            continue

        df_axe1_stats  = _read_csv_safe(os.path.join(lex_dir, f'axe1_global_stats_{lex_label}.csv'))
        df_axe1_topics = _read_csv_safe(os.path.join(lex_dir, f'axe1_topic_distribution_{lex_label}.csv'))
        df_axe2        = _read_csv_safe(os.path.join(lex_dir, f'axe2_tfidf_differential_{lex_label}.csv'))
        df_axe3_tfidf  = _read_csv_safe(os.path.join(lex_dir, f'axe3_cooc_tfidf_{lex_label}.csv'))
        df_axe3_chi2g  = _read_csv_safe(os.path.join(lex_dir, f'axe3_cooc_chi2_global_{lex_label}.csv'))
        df_axe3_words  = _read_csv_safe(os.path.join(lex_dir, f'axe3_cooc_chi2_words_{lex_label}.csv'))
        df_axe4_stats  = _read_csv_safe(os.path.join(lex_dir, f'axe4_sentiment_stats_{lex_label}.csv'))

        ws = wb.create_sheet(title=lex_label[:31])
        ws.freeze_panes = 'A3'

        ws.merge_cells('A1:F1')
        t = ws['A1']
        t.value     = f'Synthesis Q2 — Corpus : {corpus_name}  |  Lexicon : {lex_label}'
        t.font      = Font(bold=True, size=13, color='FFFFFF')
        t.fill      = PatternFill('solid', fgColor=_C_HEADER)
        t.alignment = Alignment(horizontal='center', vertical='center')
        ws.row_dimensions[1].height = 22

        for ci, h in enumerate(['Axe', 'Dimension', 'Poésie', 'Prose',
                                 'Stat clé', 'Interprétation'], start=1):
            _hcell(ws, 2, ci, h, bg=_C_SUBHDR, fc='000000')
        ws.row_dimensions[2].height = 18

        row = 3

        # 1) Topic distribution
        _section_title(ws, row, 1, 6, 'AXE 1 — Distribution des topics BERTopic')
        row += 1

        if df_axe1_stats is not None and not df_axe1_stats.empty:
            s = df_axe1_stats.iloc[0]
            chi2   = f"Chi²={s.get('chi2', float('nan')):.3f}" \
                     if pd.notna(s.get('chi2')) else 'Chi²=n/a'
            pval   = f"p={s.get('p_value', float('nan')):.4f}" \
                     if pd.notna(s.get('p_value')) else 'p=n/a'
            js     = f"JS={s.get('js_divergence', float('nan')):.4f}" \
                     if pd.notna(s.get('js_divergence')) else 'JS=n/a'
            n_p    = int(s.get('n_chunks_poetry', 0))
            n_r    = int(s.get('n_chunks_prose', 0))
            sig    = pd.notna(s.get('p_value')) and float(s.get('p_value', 1)) < 0.05

            stat_str = f'{chi2}, {pval}, {js}'
            interp   = ('Distributions significativement différentes'
                        if sig else 'Pas de différence significative')

            _dcell(ws, row, 1, 'Axe 1', bold=True)
            _dcell(ws, row, 2, 'Stats globales')
            _dcell(ws, row, 3, f'n={n_p} chunks', bg=_C_POETRY)
            _dcell(ws, row, 4, f'n={n_r} chunks', bg=_C_PROSE)
            _dcell(ws, row, 5, stat_str)
            _dcell(ws, row, 6, interp, bg=_C_SIG if sig else _C_NONSIG)
            row += 1

            if df_axe1_topics is not None and not df_axe1_topics.empty:
                df_p = df_axe1_topics[df_axe1_topics['n_poetry'] > 0]\
                           .sort_values('ratio_poetry_over_prose', ascending=False).head(3)
                df_r = df_axe1_topics[df_axe1_topics['n_prose'] > 0]\
                           .sort_values('ratio_poetry_over_prose', ascending=True).head(3)

                for rank, (_, trow) in enumerate(df_p.iterrows(), start=1):
                    tid    = int(trow['topic'])
                    ratio  = trow.get('ratio_poetry_over_prose', float('nan'))
                    kw     = _get_topic_keywords(topic_words_df, tid)
                    _dcell(ws, row, 1, '')
                    _dcell(ws, row, 2, f'Topic poésie #{rank} (id={tid})')
                    _dcell(ws, row, 3, kw, bg=_C_POETRY, wrap=True)
                    _dcell(ws, row, 4, '')
                    _dcell(ws, row, 5,
                           f"ratio={ratio:.2f}" if pd.notna(ratio) else '—')
                    _dcell(ws, row, 6, 'Sur-représenté en poésie', italic=True)
                    row += 1

                for rank, (_, trow) in enumerate(df_r.iterrows(), start=1):
                    tid   = int(trow['topic'])
                    ratio = trow.get('ratio_poetry_over_prose', float('nan'))
                    kw    = _get_topic_keywords(topic_words_df, tid)
                    _dcell(ws, row, 1, '')
                    _dcell(ws, row, 2, f'Topic prose #{rank} (id={tid})')
                    _dcell(ws, row, 3, '')
                    _dcell(ws, row, 4, kw, bg=_C_PROSE, wrap=True)
                    _dcell(ws, row, 5,
                           f"ratio={ratio:.2f}" if pd.notna(ratio) else '—')
                    _dcell(ws, row, 6, 'Sur-représenté en prose', italic=True)
                    row += 1
        else:
            _dcell(ws, row, 1, 'Axe 1')
            _dcell(ws, row, 2, 'Données absentes ou insuffisantes', bold=True)
            row += 1

        # 2) Differential TF-IDF vocabulary
        _section_title(ws, row, 1, 6, 'AXE 2 — Vocabulaire caractéristique (TF-IDF différentiel)')
        row += 1

        if df_axe2 is not None and not df_axe2.empty:
            top_p_words = df_axe2[df_axe2['characteristic_of'] == 'poetry']\
                              .head(SYNTHESE_Q2_TOP_WORDS)['word'].tolist()
            top_r_words = df_axe2[df_axe2['characteristic_of'] == 'prose']\
                              .head(SYNTHESE_Q2_TOP_WORDS)['word'].tolist()

            _dcell(ws, row, 1, 'Axe 2', bold=True)
            _dcell(ws, row, 2, f'Top {SYNTHESE_Q2_TOP_WORDS} mots')
            _dcell(ws, row, 3, ', '.join(top_p_words) if top_p_words else '—',
                   bg=_C_POETRY)
            _dcell(ws, row, 4, ', '.join(top_r_words) if top_r_words else '—',
                   bg=_C_PROSE)
            _dcell(ws, row, 5, '')
            _dcell(ws, row, 6, 'Mots absents des mots-outils, triés par différence TF-IDF',
                   italic=True)
            row += 1
        else:
            _dcell(ws, row, 1, 'Axe 2')
            _dcell(ws, row, 2, 'Données absentes ou insuffisantes', bold=True)
            row += 1

        # 3) Contextual co-occurrences
        _section_title(ws, row, 1, 6,
                       f'AXE 3 — Mots-contextes autour du lexique (fenêtre ±{COOC_WINDOW} tokens)')
        row += 1

        if df_axe3_tfidf is not None and not df_axe3_tfidf.empty:
            top_p_cooc = df_axe3_tfidf[df_axe3_tfidf['characteristic_of'] == 'poetry']\
                             .head(SYNTHESE_Q2_TOP_WORDS)['word'].tolist()
            top_r_cooc = df_axe3_tfidf[df_axe3_tfidf['characteristic_of'] == 'prose']\
                             .head(SYNTHESE_Q2_TOP_WORDS)['word'].tolist()

            _dcell(ws, row, 1, 'Axe 3', bold=True)
            _dcell(ws, row, 2, f'Top {SYNTHESE_Q2_TOP_WORDS} mots-contextes')
            _dcell(ws, row, 3, ', '.join(top_p_cooc) if top_p_cooc else '—',
                   bg=_C_POETRY)
            _dcell(ws, row, 4, ', '.join(top_r_cooc) if top_r_cooc else '—',
                   bg=_C_PROSE)
            _dcell(ws, row, 5, '')
            _dcell(ws, row, 6,
                   'Mots les plus caractéristiques dans la fenêtre contextuelle',
                   italic=True)
            row += 1

        if df_axe3_chi2g is not None and not df_axe3_chi2g.empty:
            s      = df_axe3_chi2g.iloc[0]
            chi2   = f"Chi²={s.get('chi2', float('nan')):.3f}" \
                     if pd.notna(s.get('chi2')) else 'Chi²=n/a'
            pval   = f"p={s.get('p_value', float('nan')):.4f}" \
                     if pd.notna(s.get('p_value')) else 'p=n/a'
            n_p_co = int(s.get('n_cooc_poetry', 0))
            n_r_co = int(s.get('n_cooc_prose', 0))
            sig    = pd.notna(s.get('p_value')) and float(s.get('p_value', 1)) < 0.05

            _dcell(ws, row, 1, '')
            _dcell(ws, row, 2, 'Chi² contextes')
            _dcell(ws, row, 3, f'n_cooc={n_p_co}', bg=_C_POETRY)
            _dcell(ws, row, 4, f'n_cooc={n_r_co}', bg=_C_PROSE)
            _dcell(ws, row, 5, f'{chi2}, {pval}')
            _dcell(ws, row, 6,
                   'Contextes sig. différents' if sig else 'Contextes non sig. différents',
                   bg=_C_SIG if sig else _C_NONSIG)
            row += 1

            if df_axe3_words is not None and not df_axe3_words.empty:
                top5 = df_axe3_words.head(SYNTHESE_Q2_TOP_WORDS)
                for _, wr in top5.iterrows():
                    fp = wr.get('freq_poetry', float('nan'))
                    fr = wr.get('freq_prose',  float('nan'))
                    _dcell(ws, row, 1, '')
                    _dcell(ws, row, 2, f'  {wr["word"]}')
                    _dcell(ws, row, 3,
                           f'{fp:.5f}' if pd.notna(fp) else '—', bg=_C_POETRY)
                    _dcell(ws, row, 4,
                           f'{fr:.5f}' if pd.notna(fr) else '—', bg=_C_PROSE)
                    _dcell(ws, row, 5, f'total={int(wr["total"])}')
                    _dcell(ws, row, 6, '')
                    row += 1
        else:
            _dcell(ws, row, 1, 'Axe 3')
            _dcell(ws, row, 2, 'Données absentes ou insuffisantes', bold=True)
            row += 1

        # 4) Sentiment analysis
        _section_title(ws, row, 1, 6, 'AXE 4 — Valence émotionnelle')
        row += 1

        if df_axe4_stats is not None and not df_axe4_stats.empty:
            s    = df_axe4_stats.iloc[0]
            mp   = s.get('mean_sentiment_poetry', float('nan'))
            mr   = s.get('mean_sentiment_prose',  float('nan'))
            pval = s.get('p_value',    float('nan'))
            d    = s.get('cliffs_delta', float('nan'))
            n_p  = int(s.get('n_poetry', 0))
            n_r  = int(s.get('n_prose',  0))
            sig  = pd.notna(pval) and float(pval) < 0.05

            pval_str = f'p={pval:.4f}' if pd.notna(pval) else 'p=n/a'
            d_str    = f'd={d:.3f}'    if pd.notna(d)    else 'd=n/a'

            _dcell(ws, row, 1, 'Axe 4', bold=True)
            _dcell(ws, row, 2, 'Sentiment moyen (1-5★)')
            _dcell(ws, row, 3,
                   f'{mp:.3f} (n={n_p})' if pd.notna(mp) else '—',
                   bg=_C_POETRY,
                   bold=pd.notna(mp) and pd.notna(mr) and float(mp) > float(mr))
            _dcell(ws, row, 4,
                   f'{mr:.3f} (n={n_r})' if pd.notna(mr) else '—',
                   bg=_C_PROSE,
                   bold=pd.notna(mp) and pd.notna(mr) and float(mr) > float(mp))
            _dcell(ws, row, 5, f'{pval_str}, {d_str}')
            _dcell(ws, row, 6,
                   'Différence significative' if sig else 'Pas de différence significative',
                   bg=_C_SIG if sig else _C_NONSIG)
            row += 1

            interp_d = ''
            if pd.notna(d):
                d_val = float(d)
                if   abs(d_val) < 0.147: interp_d = 'Effet négligeable'
                elif abs(d_val) < 0.330: interp_d = 'Effet petit'
                elif abs(d_val) < 0.474: interp_d = 'Effet moyen'
                else:                    interp_d = 'Effet large'
                interp_d += ' (poésie + positive)' if d_val > 0 else ' (prose + positive)'
            _dcell(ws, row, 1, '')
            _dcell(ws, row, 2, 'Taille d\'effet')
            _dcell(ws, row, 3, '')
            _dcell(ws, row, 4, '')
            _dcell(ws, row, 5, d_str)
            _dcell(ws, row, 6, interp_d, italic=True)
            row += 1
        else:
            _dcell(ws, row, 1, 'Axe 4')
            _dcell(ws, row, 2, 'Données absentes ou insuffisantes', bold=True)
            row += 1

        ws.column_dimensions['A'].width = 8
        ws.column_dimensions['B'].width = 30
        ws.column_dimensions['C'].width = 40
        ws.column_dimensions['D'].width = 40
        ws.column_dimensions['E'].width = 28
        ws.column_dimensions['F'].width = 35

        axe1_sig  = False
        axe1_js   = float('nan')
        axe3_sig  = False
        axe4_sig  = False
        axe4_d    = float('nan')
        top_p_w   = '—'
        top_r_w   = '—'
        top_p_co  = '—'
        top_r_co  = '—'

        if df_axe1_stats is not None and not df_axe1_stats.empty:
            s = df_axe1_stats.iloc[0]
            axe1_js  = s.get('js_divergence', float('nan'))
            axe1_sig = pd.notna(s.get('p_value')) and float(s.get('p_value', 1)) < 0.05
        if df_axe2 is not None and not df_axe2.empty:
            top_p_w = ', '.join(
                df_axe2[df_axe2['characteristic_of']=='poetry']
                .head(3)['word'].tolist()) or '—'
            top_r_w = ', '.join(
                df_axe2[df_axe2['characteristic_of']=='prose']
                .head(3)['word'].tolist()) or '—'
        if df_axe3_chi2g is not None and not df_axe3_chi2g.empty:
            s = df_axe3_chi2g.iloc[0]
            axe3_sig = pd.notna(s.get('p_value')) and float(s.get('p_value', 1)) < 0.05
        if df_axe3_tfidf is not None and not df_axe3_tfidf.empty:
            top_p_co = ', '.join(
                df_axe3_tfidf[df_axe3_tfidf['characteristic_of']=='poetry']
                .head(3)['word'].tolist()) or '—'
            top_r_co = ', '.join(
                df_axe3_tfidf[df_axe3_tfidf['characteristic_of']=='prose']
                .head(3)['word'].tolist()) or '—'
        if df_axe4_stats is not None and not df_axe4_stats.empty:
            s = df_axe4_stats.iloc[0]
            axe4_d   = s.get('cliffs_delta', float('nan'))
            axe4_sig = pd.notna(s.get('p_value')) and float(s.get('p_value', 1)) < 0.05

        global_rows.append({
            'lexique':           lex_label,
            'axe1_JS':           round(float(axe1_js), 4) if pd.notna(axe1_js) else None,
            'axe1_sig':          'Oui' if axe1_sig else 'Non',
            'axe2_top_poesie':   top_p_w,
            'axe2_top_prose':    top_r_w,
            'axe3_sig':          'Oui' if axe3_sig else 'Non',
            'axe3_top_poesie':   top_p_co,
            'axe3_top_prose':    top_r_co,
            'axe4_cliffs_delta': round(float(axe4_d), 3) if pd.notna(axe4_d) else None,
            'axe4_sig':          'Oui' if axe4_sig else 'Non',
        })

    # Global synthesis
    if global_rows:
        ws_g = wb.create_sheet(title='Vue globale', index=0)
        ws_g.freeze_panes = 'A3'

        ws_g.merge_cells('A1:J1')
        t = ws_g['A1']
        t.value     = f'Vue globale Q2 — {corpus_name} — tous lexiques'
        t.font      = Font(bold=True, size=13, color='FFFFFF')
        t.fill      = PatternFill('solid', fgColor=_C_HEADER)
        t.alignment = Alignment(horizontal='center', vertical='center')
        ws_g.row_dimensions[1].height = 22

        headers = [
            'Lexique',
            'Axe1 JS-div', 'Axe1 sig.',
            'Axe2 top poésie (3)', 'Axe2 top prose (3)',
            'Axe3 sig.',
            'Axe3 top poésie (3)', 'Axe3 top prose (3)',
            'Axe4 Cliff\'s d', 'Axe4 sig.',
        ]
        for ci, h in enumerate(headers, start=1):
            _hcell(ws_g, 2, ci, h, bg=_C_SUBHDR, fc='000000')
        ws_g.row_dimensions[2].height = 18

        for ri, gr in enumerate(global_rows, start=3):
            bg = _C_ALT if ri % 2 == 0 else None
            ws_g.cell(row=ri, column=1,  value=gr['lexique']).font = Font(bold=True)
            ws_g.cell(row=ri, column=2,  value=gr['axe1_JS'])
            c_sig1 = ws_g.cell(row=ri, column=3,  value=gr['axe1_sig'])
            c_sig1.fill = PatternFill('solid', fgColor=_C_SIG if gr['axe1_sig']=='Oui' else _C_NONSIG)
            ws_g.cell(row=ri, column=4,  value=gr['axe2_top_poesie']).fill = \
                PatternFill('solid', fgColor=_C_POETRY)
            ws_g.cell(row=ri, column=5,  value=gr['axe2_top_prose']).fill = \
                PatternFill('solid', fgColor=_C_PROSE)
            c_sig3 = ws_g.cell(row=ri, column=6,  value=gr['axe3_sig'])
            c_sig3.fill = PatternFill('solid', fgColor=_C_SIG if gr['axe3_sig']=='Oui' else _C_NONSIG)
            ws_g.cell(row=ri, column=7,  value=gr['axe3_top_poesie']).fill = \
                PatternFill('solid', fgColor=_C_POETRY)
            ws_g.cell(row=ri, column=8,  value=gr['axe3_top_prose']).fill = \
                PatternFill('solid', fgColor=_C_PROSE)
            ws_g.cell(row=ri, column=9,  value=gr['axe4_cliffs_delta'])
            c_sig4 = ws_g.cell(row=ri, column=10, value=gr['axe4_sig'])
            c_sig4.fill = PatternFill('solid', fgColor=_C_SIG if gr['axe4_sig']=='Oui' else _C_NONSIG)
            for ci in range(1, 11):
                ws_g.cell(row=ri, column=ci).border    = _bdr()
                ws_g.cell(row=ri, column=ci).alignment = \
                    Alignment(horizontal='left', vertical='center', wrap_text=True)
                if bg and ws_g.cell(row=ri, column=ci).fill.fgColor.rgb in ('00000000', ''):
                    ws_g.cell(row=ri, column=ci).fill = PatternFill('solid', fgColor=bg)

        col_widths = [18, 12, 10, 35, 35, 10, 35, 35, 14, 10]
        for ci, w in enumerate(col_widths, start=1):
            ws_g.column_dimensions[get_column_letter(ci)].width = w

    out_path = os.path.join(OUT_DIR_ROOT, corpus_name,
                            f'synthese_Q2_{corpus_name}.xlsx')
    wb.save(out_path)
    print(f'  [Synthesis Q2] Exported to : {out_path}')
    return out_path


print('\nGenerating Q2 synthesis...')
for corpus_cfg in CORPUS_CONFIGS:
    build_q2_summary_for_corpus(corpus_cfg['name'])
print('\n Synthesis Q2 done.')


Generating Q2 synthesis...
  [Synthesis Q2] Exported to : ./final_results/lesbiennes/synthese_Q2_lesbiennes.xlsx
  [Synthesis Q2 | f_hetero] File analyse_differentielle absent, skip.
  [Synthesis Q2 | homo_m] File analyse_differentielle absent, skip.
  [Synthesis Q2 | h_hetero] File analyse_differentielle absent, skip.

 Synthesis Q2 done.
